In [0]:
from pyspark.sql.functions import (
    col,
    count,
    sum,
    avg,
    round,
    when
)

FACT_TABLE = "workspace.default.fact_trip"
LOCATION_TABLE = "workspace.default.dim_location"

df_fact = spark.table(FACT_TABLE)
df_location = spark.table(LOCATION_TABLE)

print("Fact rows:", df_fact.count())
print("Location rows:", df_location.count())

In [0]:
df_location_enriched = (
    df_fact
    .join(
        df_location.select(
            col("location_key").alias("pickup_location_key"),
            col("location_id"),
            col("borough"),
            col("zone"),
            col("service_zone")
        ),
        on="pickup_location_key",
        how="left"
    )
)

In [0]:
display(
    df_location_enriched.select(
        "pickup_location_key",
        "location_id",
        "borough",
        "zone",
        "service_zone",
        "trip_miles",
        "base_passenger_fare",
        "driver_pay"
    ).limit(10)
)

In [0]:
df_location_gold = (
    df_location_enriched
    .groupBy(
        "pickup_location_key",
        "location_id",
        "borough",
        "zone",
        "service_zone"
    )
    .agg(
        count("*").alias("total_trips"),

        sum("trip_miles").alias(
            "total_trip_miles"
        ),

        avg("trip_miles").alias(
            "avg_trip_distance"
        ),

        avg("calculated_trip_time_seconds").alias(
            "avg_trip_duration_seconds"
        ),

        avg("customer_wait_seconds").alias(
            "avg_customer_wait_seconds"
        ),

        sum("base_passenger_fare").alias(
            "total_passenger_fare"
        ),

        sum("tips").alias(
            "total_tips"
        ),

        sum("driver_pay").alias(
            "total_driver_pay"
        )
    )
)

In [0]:
df_location_gold = (
    df_location_gold
    .withColumn(
        "avg_trip_distance",
        round(col("avg_trip_distance"), 2)
    )
    .withColumn(
        "avg_trip_duration_minutes",
        round(
            col("avg_trip_duration_seconds") / 60,
            2
        )
    )
    .withColumn(
        "avg_customer_wait_minutes",
        round(
            col("avg_customer_wait_seconds") / 60,
            2
        )
    )
)

In [0]:
df_location_gold = df_location_gold.drop(
    "avg_trip_duration_seconds",
    "avg_customer_wait_seconds"
)

In [0]:
df_location_gold = (
    df_location_gold
    .withColumn(
        "fare_per_mile",
        round(
            when(
                col("total_trip_miles") > 0,
                col("total_passenger_fare")
                / col("total_trip_miles")
            ),
            2
        )
    )
)

In [0]:
print(
    "Gold location rows:",
    df_location_gold.count()
)

print(
    "Total trips:",
    df_location_gold
    .select(sum("total_trips"))
    .collect()[0][0]
)

In [0]:
GOLD_LOCATION_TABLE = "workspace.default.gold_location_metrics"

(
    df_location_gold
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_LOCATION_TABLE)
)

In [0]:
df_location_final = spark.table(
    "workspace.default.gold_location_metrics"
)

print(
    "Persisted rows:",
    df_location_final.count()
)

print(
    "Persisted trips:",
    df_location_final
    .select(sum("total_trips"))
    .collect()[0][0]
)

In [0]:
display(
    df_location_final
    .orderBy(
        col("total_trips").desc()
    )
    .limit(20)
)